# Bernstein–Vazirani hidden string

Recover a hidden binary string from one coherent oracle query.

The SDK reference and MettleQ calls below use the same circuit and result contract. Timing includes the complete call shown.

In [1]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.primitives import StatevectorEstimator, StatevectorSampler

from mettleq.integrations.qiskit import (
    MettleQBackend,
    MettleQEstimatorV2,
    MettleQSamplerV2,
)
from tutorials._support import (
    benchmark,
    emit_result,
    max_abs_error,
    phase_aligned_statevector_error,
    qiskit_selection,
    total_variation_distance,
)

In [2]:
secret = "101101"
n = len(secret)
circuit = QuantumCircuit(n + 1)
circuit.x(n)
circuit.h(range(n + 1))
for wire, bit in enumerate(reversed(secret)):
    if bit == "1":
        circuit.cx(wire, n)
circuit.h(range(n))

def get_reference():
    return Statevector.from_instruction(circuit).probabilities(qargs=range(n))

reference, reference_ms, _ = benchmark(get_reference)
backend = MettleQBackend(method="statevector", device="cpu")
compiled = transpile(circuit, backend, optimization_level=1)

def get_mettleq():
    state = backend.run(compiled, shots=1, return_statevector=True).result().data(0)["statevector"]
    return Statevector(state).probabilities(qargs=range(n))

candidate, mettleq_ms, _ = benchmark(get_mettleq)
error = max_abs_error(reference, candidate)
recovered = format(int(np.argmax(candidate)), f"0{n}b")
method, device = qiskit_selection(backend)
tutorial_result = emit_result(
    notebook="qiskit/06_bernstein_vazirani.ipynb",
    framework="qiskit",
    reference_ms=reference_ms,
    mettleq_ms=mettleq_ms,
    check="probability vector atol=2e-6 and exact hidden string",
    passed=error <= 2e-6 and recovered == secret,
    exact_match=recovered == secret,
    selected_method=method,
    selected_device=device,
    metrics={"max_probability_error": error, "secret": secret, "recovered": recovered},
)

TUTORIAL_RESULT::{"check": "probability vector atol=2e-6 and exact hidden string", "exact_match": true, "framework": "qiskit", "machine": "arm64", "metrics": {"max_probability_error": 2.0281592372217716e-07, "recovered": "101101", "secret": "101101"}, "mettleq_median_ms": 0.7220409752335399, "notebook": "qiskit/06_bernstein_vazirani.ipynb", "notes": "", "passed": true, "python": "3.13.2", "reference_median_ms": 0.3286669962108135, "reference_over_mettleq": 0.45519161305839756, "schema_version": 1, "selected_device": "cpu", "selected_method": "statevector"}
